# MuSiQue · build the questions (local)

Writes `questions.pkl`: MuSiQue 2-hop questions, each with its 2 supporting paragraphs and 16 distractor
paragraphs turned into facts by Llama-3.1-70B, in a fixed nested order. Every notebook in this folder reads it.
The steps and filters are described at the top of `common/build.py`. Cost: about $1 of 70B calls.

**Run once.** If `questions.pkl` exists, this notebook only shows it; set `REBUILD = True` to build a new set (then
move `results/` away: answers to an old question set are never mixed with a new one).

In [1]:
import os, pickle, sys
sys.path.insert(0, "../common")

DATASET = "MuSiQue"
N_QUESTIONS = 500
K_MAX = 16            # distractor paragraphs per question (k = 0, 4, 8, 16)
REBUILD = True
EXTEND = True         # keep the questions already in questions.pkl and add new ones up to N_QUESTIONS

In [2]:
if os.path.exists("questions.pkl") and not REBUILD:
    print("questions.pkl already exists: set REBUILD = True to build a new set")
else:
    from build import build
    build(DATASET, N_QUESTIONS, K_MAX, extend=EXTEND and os.path.exists("questions.pkl"))

/Users/mina/miniforge3/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Extending questions.pkl: 200 questions kept, 300 to build
1. validation: 1252 2-hop questions, 367 fresh with at least 16 distractors


Extracting facts: 100%|██████████| 760/760 [07:00<00:00,  1.81it/s]


   validation: {'screened': 367, 'unknown': 177, 'answerable': 97, 'selected': 38, 'complete': 38} → 38 questions so far
1. train: 14376 2-hop questions, 3320 fresh with at least 16 distractors


Extracting facts: 100%|██████████| 140/140 [01:24<00:00,  1.65it/s]


   train: {'screened': 400, 'unknown': 259, 'answerable': 202, 'selected': 7, 'complete': 7} → 45 questions so far


... (12 outputs left out)

Extracting facts: 100%|██████████| 940/940 [06:11<00:00,  2.53it/s]


   train: {'screened': 3200, 'unknown': 2063, 'answerable': 1626, 'selected': 255, 'complete': 255} → 293 questions so far


Extracting facts: 100%|██████████| 180/180 [01:18<00:00,  2.29it/s]


   train: {'screened': 3320, 'unknown': 2140, 'answerable': 1686, 'selected': 264, 'complete': 262} → 300 questions so far
Saved 500 questions to questions.pkl {'validation': 104, 'train': 396}. API: {'model': 'meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo', 'requests': 15491, 'failed': 0, 'input_tokens': 2526039, 'output_tokens': 982112, 'cost_usd': 1.4033}


In [3]:
from collections import Counter
questions = pickle.load(open("questions.pkl", "rb"))
print(f"{len(questions)} questions · sources {dict(Counter(q['source_split'] for q in questions))} · "
      f"{len({q['answer'] for q in questions})} distinct answers")
for q in questions[:5]:
    supporting = sum(len(p["facts"]) for p in q["layout"] if p["is_supporting"])
    distractor = sum(len(p["facts"]) for p in q["layout"] if not p["is_supporting"])
    print(f"- {q['question']}  →  {q['answer']}   ({supporting} supporting facts, {distractor} in distractors)")

500 questions · sources {'validation': 104, 'train': 396} · 500 distinct answers
- Who followed the father of Menucha Rochel Slonim?  →  Menachem Mendel Schneersohn   (10 supporting facts, 108 in distractors)
- As a result of the black death it is estimated that the population of the country for Bâton à feu was reduced by how much?  →  30% to 65%   (13 supporting facts, 203 in distractors)
- When does the show of the cast member of Politically Incorrect start again in 2018?  →  January 19, 2018   (63 supporting facts, 154 in distractors)
- Where is Amílcar Cabral International Airport?  →  central Atlantic Ocean   (17 supporting facts, 138 in distractors)
- What season was the performer of Addicted a part of?  →  season five   (14 supporting facts, 132 in distractors)
